In [ ]:
# Prueba Scraping PcComponentes con Playwright
# Usamos Playwright porque la web carga con JavaScript y tiene protección antibot.

SyntaxError: invalid syntax (2544792687.py, line 2)

In [13]:
from playwright.async_api import async_playwright
import pandas as pd
from datetime import datetime

In [14]:
async def scrape_pccomponentes():
    productos = []
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        await page.goto(
            "https://www.pccomponentes.com/portatiles")
        
        # Espera a que carguen los productos
        await page.wait_for_selector(
            "h3.product-card__title", 
            timeout=15000)
        
        nombres = await page.query_selector_all(
                    "h3.product-card__title")
        precios = await page.query_selector_all(
                    "[data-e2e='price-card']")
        precios_orig = await page.query_selector_all(
                    "[data-e2e='crossedPrice']")
        
        print(f"Nombres encontrados: {len(nombres)}")
        print(f"Precios encontrados: {len(precios)}")
        
        for i, nombre in enumerate(nombres):
            precio_actual = (
                await precios[i].inner_text() 
                if i < len(precios) else None)
            precio_original = (
                await precios_orig[i].inner_text() 
                if i < len(precios_orig) else None)
            
            productos.append({
                "nombre": await nombre.inner_text(),
                "precio_actual": precio_actual,
                "precio_original": precio_original,
                "plataforma": "PcComponentes",
                "fecha": datetime.now().strftime(
                            "%Y-%m-%d %H:%M")
            })
        
        await browser.close()
    
    return pd.DataFrame(productos)

In [24]:
from playwright_stealth import Stealth

async def diagnostico_stealth():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,  # Visible para ver qué pasa
            args=["--disable-blink-features=AutomationControlled"]
        )
        
        context = await browser.new_context(
            viewport={"width": 1280, "height": 800},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            locale="es-ES",
        )
        
        # API nueva de playwright-stealth
        stealth = Stealth()
        await stealth.apply_stealth_async(context)
        
        page = await context.new_page()
        
        await page.goto(
            "https://www.pccomponentes.com/portatiles",
            wait_until="domcontentloaded",
            timeout=30000)
        
        # Esperamos más tiempo simulando comportamiento humano
        await page.wait_for_timeout(8000)
        
        html = await page.content()
        
        print(f"Tamaño del HTML: {len(html)} caracteres")
        print(f"¿Contiene 'product-card': {'product-card' in html}")
        print(f"¿Contiene 'Portátil': {'Portátil' in html}")
        print(f"¿Contiene 'robot': {'robot' in html.lower()}")
        
        with open("../data/raw/pagina_pccomponentes.html", "w", encoding="utf-8") as f:
            f.write(html)
        print("HTML guardado en data/raw/pagina_pccomponentes.html")
        
        await browser.close()

await diagnostico_stealth()

Tamaño del HTML: 1196018 caracteres
¿Contiene 'product-card': True
¿Contiene 'Portátil': True
¿Contiene 'robot': True
HTML guardado en data/raw/pagina_pccomponentes.html


In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
from pathlib import Path

def scrape_idealo():
    url = "https://www.idealo.es/cat/3394/portatiles.html"
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "es-ES,es;q=0.9",
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
    except requests.RequestException as e:
        print(f"Error de conexión con Idealo: {e}")
        return pd.DataFrame()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_dir = Path("../data/raw")
    raw_dir.mkdir(parents=True, exist_ok=True)
    html_path = raw_dir / f"idealo_portatiles_{timestamp}.html"
    html_path.write_text(response.text, encoding="utf-8")

    print(f"Status: {response.status_code}")
    print(f"Tamaño HTML: {len(response.text)}")
    print(f"HTML guardado en: {html_path}")

    texto_lower = response.text.lower()
    bloqueado = any(k in texto_lower for k in ["akamai", "sorry", "reference id", "captcha", "forbidden", "access denied"])
    print(f"¿Posible bloqueo anti-bot?: {bloqueado}")

    if response.status_code != 200 or bloqueado:
        print("Plan D bloqueado en este entorno/red. Se recomienda estrategia alternativa.")
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, "html.parser")
    items = soup.select("article, .offerList-item, .sr-resultList__item")
    productos = []

    for item in items:
        nombre = item.get_text(" ", strip=True)[:160]
        precio_tag = item.select_one("[class*='price'], [data-testid*='price']")
        precio = precio_tag.get_text(" ", strip=True) if precio_tag else None

        if nombre and precio:
            productos.append({
                "nombre": nombre,
                "precio_actual": precio,
                "plataforma": "Idealo",
                "fecha": datetime.now().strftime("%Y-%m-%d %H:%M")
            })

    df = pd.DataFrame(productos)
    print(f"Productos extraídos: {len(df)}")
    return df

df_idealo = scrape_idealo()
df_idealo.head()

Error de conexión con Idealo: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


""


In [25]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from scraper import run_daily_scrape

# Ejecución diaria (ajusta headless=False si quieres ver el navegador)
df_diario = await run_daily_scrape(max_items_per_platform=20, headless=True)
df_diario.head()

Archivo guardado: /Users/david/Documents/scraper/data/raw/precios_portatiles_20260330_1251.csv
plataforma
PcComponentes    20
Amazon           20
Name: count, dtype: int64


,nombre,precio_actual,precio_original,descuento,valoracion,plataforma,fecha
0,Portátil Alurin Flex Advance AMD Ryzen 7-5825U...,"489,99€","644,99€",None,NaN,PcComponentes,2026-03-30 12:51
1,"PcCom Revolt 5070 Intel Core i7-14650HX 16""/QH...","1.658,99€","2.079,99€",None,NaN,PcComponentes,2026-03-30 12:51
2,"Portátil Lenovo LOQ 15IRX10 15.6"" Intel Core i...",1.149€,1.499€,None,NaN,PcComponentes,2026-03-30 12:51
3,"Portátil Lenovo Legion 5 15AHP10 15.1"" AMD Ryz...",1.289€,1.699€,None,NaN,PcComponentes,2026-03-30 12:51
4,Apple Macbook Air Apple M4 10 Núcleos/16 GB/25...,899€,1.179€,None,NaN,PcComponentes,2026-03-30 12:51
